# MADDPG Agent Implementation

## Version 1

In [3]:
!pip install unityagents

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 42.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of unityagents to determine which version is compatible with other requirements. This could take a while.
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
import os

shared_drives_path = '/content/drive/Shareddrives'
print(os.listdir(shared_drives_path))  # Lists all shared drives you have access to


['Hydration Automation (HA)']


In [1]:
import numpy as np
import random
from collections import namedtuple, deque
import copy
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Actor Model
class Actor(nn.Module):
    def __init__(self, state_size, action_size, seed, fc1_units=128, fc2_units=128):
        super(Actor, self).__init__()
        self.seed = torch.manual_seed(seed)
        self.fc1 = nn.Linear(state_size, fc1_units)
        self.fc2 = nn.Linear(fc1_units, fc2_units)
        self.fc3 = nn.Linear(fc2_units, action_size)
        self.tanh = nn.Tanh()

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        x = torch.relu(self.fc2(x))
        return self.tanh(self.fc3(x))

# Critic Model (centralized: takes all states and all actions)
class Critic(nn.Module):
    def __init__(self, full_state_size, full_action_size, seed, fcs1_units=128, fc2_units=128):
        super(Critic, self).__init__()
        self.seed = torch.manual_seed(seed)
        self.fcs1 = nn.Linear(full_state_size, fcs1_units)
        self.fc2 = nn.Linear(fcs1_units + full_action_size, fc2_units)
        self.fc3 = nn.Linear(fc2_units, 1)

    def forward(self, states, actions):
        xs = torch.relu(self.fcs1(states))
        x = torch.cat((xs, actions), dim=1)
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

# Replay Buffer (shared)
class ReplayBuffer:
    def __init__(self, buffer_size, batch_size, seed):
        self.memory = deque(maxlen=buffer_size)
        self.batch_size = batch_size
        self.experience = namedtuple("Experience", field_names=[
            "states", "actions", "rewards", "next_states", "dones"
        ])
        self.seed = random.seed(seed)

    def add(self, states, actions, rewards, next_states, dones):
        e = self.experience(states, actions, rewards, next_states, dones)
        self.memory.append(e)

    def sample(self):
        experiences = random.sample(self.memory, k=self.batch_size)
        states = torch.from_numpy(np.vstack([e.states for e in experiences])).float().to(device)
        actions = torch.from_numpy(np.vstack([e.actions for e in experiences])).float().to(device)
        rewards = torch.from_numpy(np.vstack([e.rewards for e in experiences])).float().to(device)
        next_states = torch.from_numpy(np.vstack([e.next_states for e in experiences])).float().to(device)
        dones = torch.from_numpy(np.vstack([e.dones for e in experiences]).astype(np.uint8)).float().to(device)
        return (states, actions, rewards, next_states, dones)

    def __len__(self):
        return len(self.memory)

# Ornstein-Uhlenbeck Noise
class OUNoise:
    def __init__(self, size, seed, mu=0., theta=0.15, sigma=0.2):
        self.mu = mu * np.ones(size)
        self.theta = theta
        self.sigma = sigma
        self.seed = random.seed(seed)
        self.size = size
        self.reset()

    def reset(self):
        self.state = copy.copy(self.mu)

    def sample(self):
        dx = self.theta * (self.mu - self.state) + self.sigma * np.random.randn(self.size)
        self.state += dx
        return self.state

# Single DDPG agent (for MADDPG)
class Agent:
    def __init__(self, idx, state_size, action_size, num_agents, random_seed):
        self.idx = idx
        self.state_size = state_size
        self.action_size = action_size
        self.num_agents = num_agents
        self.seed = random_seed

        self.actor_local = Actor(state_size, action_size, random_seed).to(device)
        self.actor_target = Actor(state_size, action_size, random_seed).to(device)
        self.actor_optimizer = optim.Adam(self.actor_local.parameters(), lr=1e-4)

        # Centralized critic: input is ALL agents' states and ALL agents' actions
        self.critic_local = Critic(state_size*num_agents, action_size*num_agents, random_seed).to(device)
        self.critic_target = Critic(state_size*num_agents, action_size*num_agents, random_seed).to(device)
        self.critic_optimizer = optim.Adam(self.critic_local.parameters(), lr=1e-3)

        self.noise = OUNoise(action_size, random_seed)
        self.hard_update(self.actor_target, self.actor_local)
        self.hard_update(self.critic_target, self.critic_local)

    def act(self, state, noise_weight=1.0):
        state = torch.from_numpy(state).float().to(device)
        self.actor_local.eval()
        with torch.no_grad():
            action = self.actor_local(state).cpu().data.numpy()
        self.actor_local.train()
        action += self.noise.sample() * noise_weight
        return np.clip(action, -1, 1)

    def reset(self):
        self.noise.reset()

    def hard_update(self, target, source):
        for t, s in zip(target.parameters(), source.parameters()):
            t.data.copy_(s.data)

    def soft_update(self, local, target, tau):
        for target_param, local_param in zip(target.parameters(), local.parameters()):
            target_param.data.copy_(tau*local_param.data + (1.0-tau)*target_param.data)


class MADDPG:
    def __init__(self, state_size, action_size, num_agents, random_seed):
        self.num_agents = num_agents
        self.state_size = state_size
        self.action_size = action_size
        self.agents = [Agent(i, state_size, action_size, num_agents, random_seed+i) for i in range(num_agents)]
        self.memory = ReplayBuffer(buffer_size=int(1e5), batch_size=128, seed=random_seed)
        self.tau = 1e-3
        self.gamma = 0.99

    def reset(self):
        for agent in self.agents:
            agent.reset()

    def act(self, states, noise_weight=1.0):
        actions = [agent.act(state, noise_weight) for agent, state in zip(self.agents, states)]
        return np.array(actions)

    def step(self, states, actions, rewards, next_states, dones):
        # Store in shared buffer: concatenate all states/actions for critic
        self.memory.add(np.concatenate(states),
                        np.concatenate(actions),
                        rewards,
                        np.concatenate(next_states),
                        dones)
        if len(self.memory) > self.memory.batch_size:
            for agent_idx in range(self.num_agents):
                experiences = self.memory.sample()
                self.learn(experiences, agent_idx)

    def learn(self, experiences, agent_idx):
        states, actions, rewards, next_states, dones = experiences

        num_agents = self.num_agents
        state_size = self.state_size
        action_size = self.action_size

        # Current agent selects their own actor and critic
        agent = self.agents[agent_idx]

        # -- Get actions/next actions for all agents
        # Split states/actions into separate agents
        states_all = states.view(-1, num_agents, state_size)
        next_states_all = next_states.view(-1, num_agents, state_size)
        actions_all = actions.view(-1, num_agents, action_size)

        # Next actions using target actors (for all agents)
        next_actions = []
        for i, a in enumerate(self.agents):
            a_next = a.actor_target(next_states_all[:, i, :])
            next_actions.append(a_next)
        next_actions = torch.cat(next_actions, dim=1)  # (batch, num_agents * action_size)

        # Target critic: centralized, using ALL next_states and next_actions
        Q_targets_next = agent.critic_target(
            next_states.view(-1, num_agents * state_size),
            next_actions
        )

        # Only use rewards/dones for this agent
        agent_rewards = rewards[:, agent_idx].unsqueeze(1)
        agent_dones = dones[:, agent_idx].unsqueeze(1)

        Q_targets = agent_rewards + (self.gamma * Q_targets_next * (1 - agent_dones))

        # Critic loss (uses all states, all actions)
        Q_expected = agent.critic_local(
            states.view(-1, num_agents * state_size),
            actions.view(-1, num_agents * action_size)
        )
        critic_loss = nn.MSELoss()(Q_expected, Q_targets.detach())
        agent.critic_optimizer.zero_grad()
        critic_loss.backward()
        torch.nn.utils.clip_grad_norm_(agent.critic_local.parameters(), 1)
        agent.critic_optimizer.step()

        # Actor loss: current agent's actor, others' actions fixed
        actions_pred = []
        for i, a in enumerate(self.agents):
            if i == agent_idx:
                a_pred = a.actor_local(states_all[:, i, :])
            else:
                a_pred = a.actor_local(states_all[:, i, :]).detach()
            actions_pred.append(a_pred)
        actions_pred = torch.cat(actions_pred, dim=1)

        actor_loss = -agent.critic_local(
            states.view(-1, num_agents * state_size),
            actions_pred
        ).mean()

        agent.actor_optimizer.zero_grad()
        actor_loss.backward()
        agent.actor_optimizer.step()

        # Soft update
        agent.soft_update(agent.critic_local, agent.critic_target, self.tau)
        agent.soft_update(agent.actor_local, agent.actor_target, self.tau)


from unityagents import UnityEnvironment

env = UnityEnvironment(file_name="Tennis.app")
brain_name = env.brain_names[0]
env_info = env.reset(train_mode=True)[brain_name]
num_agents = len(env_info.agents)
state_size = env_info.vector_observations.shape[1]
action_size = env_info.vector_observations.shape[1]  # Should be brain.vector_action_space_size

maddpg = MADDPG(state_size, action_size, num_agents, random_seed=0)

def train_maddpg(n_episodes=400, max_t=1000, goal_score=0.5):
    scores_deque = deque(maxlen=100)
    scores = []
    print_every = 10

    for i_episode in range(1, n_episodes+1):
        env_info = env.reset(train_mode=True)[brain_name]
        states = env_info.vector_observations
        maddpg.reset()
        scores_agents = np.zeros(num_agents)
        for t in range(max_t):
            actions = maddpg.act(states, noise_weight=1.0)
            env_info = env.step(actions)[brain_name]
            next_states = env_info.vector_observations
            rewards = np.array(env_info.rewards)
            dones = np.array(env_info.local_done, dtype=np.uint8)
            maddpg.step(states, actions, rewards, next_states, dones)
            states = next_states
            scores_agents += rewards
            if np.any(dones):
                break
        max_score = np.max(scores_agents)
        scores_deque.append(max_score)
        scores.append(max_score)
        print('\rEpisode {}\tMax Score: {:.3f}\tAvg100: {:.3f}'.format(
            i_episode, max_score, np.mean(scores_deque)), end="")
        if i_episode % print_every == 0:
            print('\rEpisode {}\tMax Score: {:.3f}\tAvg100: {:.3f}'.format(
                i_episode, max_score, np.mean(scores_deque)))
        if np.mean(scores_deque) >= goal_score:
            print('\nEnvironment solved in {:d} episodes!\tAverage Score: {:.2f}'.format(
                i_episode, np.mean(scores_deque)))
            # Save all agent weights
            for idx, agent in enumerate(maddpg.agents):
                torch.save(agent.actor_local.state_dict(), f"actor_agent{idx}.pth")
                torch.save(agent.critic_local.state_dict(), f"critic_agent{idx}.pth")
            break
    return scores

scores = train_maddpg(n_episodes=500, max_t=1000, goal_score=0.5)

env.close()

plt.figure(figsize=(10,5))
plt.plot(np.arange(1, len(scores)+1), scores)
plt.xlabel('Episode')
plt.ylabel('Max Score (per episode)')
plt.title('MADDPG Multi-Agent Training Progress')
plt.grid(True)
plt.show()


ModuleNotFoundError: No module named 'unityagents'